In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

loose_scripts = ["eval_whisper_spectrogram.py", "train_whisper_backend.py"]

In [3]:
import os

DRIVE_DIR = "/content/drive/MyDrive/deepfake audio detection"
WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

zips_to_extract = [
    "aasist.zip",
    "aasist_urdu_prepared_v2.zip",
    "csalt_openvoice_test_prepared.zip",
    "test_audio_prepared.zip",
    "test_mms_prepared.zip",
    "exp_urdu_full_v2.zip",
    "exp_urdu_vc_included.zip",
    "exp_ssl_backend_full.zip",
    "exp_cnn_spectrogram_full.zip",
    "exp_whisper_backend_full.zip",
]

for z in zips_to_extract:
    src = f'"{DRIVE_DIR}/{z}"'
    print(f"Extracting {z}...")
    !unzip -q {src} -d "{WORK_DIR}"

# copy loose scripts needed for imports
loose_scripts = ["eval_cnn_spectrogram.py", "eval_ssl_backend.py",
                  "train_cnn_spectrogram.py", "train_ssl_backend.py",
                  "eval_whisper_backend.py", "train_whisper_backend.py","detect_single.py"]
for s in loose_scripts:
    !cp "{DRIVE_DIR}/{s}" "{WORK_DIR}/"
!ls "{WORK_DIR}"

Extracting aasist.zip...
Extracting aasist_urdu_prepared_v2.zip...
Extracting csalt_openvoice_test_prepared.zip...
Extracting test_audio_prepared.zip...
Extracting test_mms_prepared.zip...
Extracting exp_urdu_full_v2.zip...
Extracting exp_urdu_vc_included.zip...
Extracting exp_ssl_backend_full.zip...
Extracting exp_cnn_spectrogram_full.zip...
Extracting exp_whisper_backend_full.zip...
aasist			       exp_urdu_full_v2
aasist_urdu_prepared_v2        exp_urdu_vc_included
csalt_openvoice_test_prepared  exp_whisper_backend_full
detect_single.py	       test_audio_prepared
eval_cnn_spectrogram.py        test_mms_prepared
eval_ssl_backend.py	       train_cnn_spectrogram.py
eval_whisper_backend.py        train_ssl_backend.py
exp_cnn_spectrogram_full       train_whisper_backend.py
exp_ssl_backend_full


In [ ]:
%cd /content/work
!pip install -q scikit-learn transformers librosa soundfile

/content/work


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


In [ ]:
import subprocess
import re

os.chdir("/content/work")

# each entry: (label, command_list)
tests = []

aasist_configs = {
    "AASIST-L Zero-shot": "aasist/models/weights/AASIST-L.pth",
    "AASIST-L Fine-tuned (TTS)": "exp_urdu_full_v2/best.pth",
    "AASIST-L Fine-tuned (TTS+VC)": "exp_urdu_vc_included/best.pth",
}

datasets = {
    "CSALT (TTS)": ("aasist_urdu_prepared_v2/protocol_test.txt", "aasist_urdu_prepared_v2"),
    "OpenVoice (own test_audio)": ("test_audio_prepared/protocol_test_audio.txt", "test_audio_prepared"),
    "OpenVoice (CSALT-converted)": ("csalt_openvoice_test_prepared/protocol_csalt_ov_test.txt", "csalt_openvoice_test_prepared"),
    "MMS-TTS + own recordings": ("test_mms_prepared/protocol_mms_test.txt", "test_mms_prepared"),
}

results = {}

def run_aasist_eval(model_path, protocol, base_dir):
    cmd = [
        "python", "aasist/eval_urdu.py",
        "--config", "aasist/config/AASIST-L.conf",
        "--protocol", protocol,
        "--base_dir", base_dir,
        "--model_path", model_path,
    ]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    match = re.search(r"EER:\s*([\d.]+)%", out)
    return float(match.group(1)) if match else None

for model_label, model_path in aasist_configs.items():
    for ds_label, (protocol, base_dir) in datasets.items():
        print(f"Running: {model_label} on {ds_label}...")
        eer = run_aasist_eval(model_path, protocol, base_dir)
        results[(model_label, ds_label)] = eer
        print(f"  -> EER: {eer}%")

print("\nAASIST runs complete.")

Running: AASIST-L Zero-shot on CSALT (TTS)...
  -> EER: 16.09%
Running: AASIST-L Zero-shot on OpenVoice (own test_audio)...
  -> EER: 59.38%
Running: AASIST-L Zero-shot on OpenVoice (CSALT-converted)...
  -> EER: 74.6%
Running: AASIST-L Zero-shot on MMS-TTS + own recordings...
  -> EER: 51.46%
Running: AASIST-L Fine-tuned (TTS) on CSALT (TTS)...
  -> EER: 23.98%
Running: AASIST-L Fine-tuned (TTS) on OpenVoice (own test_audio)...
  -> EER: 71.88%
Running: AASIST-L Fine-tuned (TTS) on OpenVoice (CSALT-converted)...
  -> EER: 59.37%
Running: AASIST-L Fine-tuned (TTS) on MMS-TTS + own recordings...
  -> EER: 61.04%
Running: AASIST-L Fine-tuned (TTS+VC) on CSALT (TTS)...
  -> EER: 23.61%
Running: AASIST-L Fine-tuned (TTS+VC) on OpenVoice (own test_audio)...
  -> EER: 84.38%
Running: AASIST-L Fine-tuned (TTS+VC) on OpenVoice (CSALT-converted)...
  -> EER: 59.31%
Running: AASIST-L Fine-tuned (TTS+VC) on MMS-TTS + own recordings...
  -> EER: 74.17%

AASIST runs complete.


In [ ]:
def run_generic_eval(script, model_path, protocol, base_dir):
    cmd = ["python", script, "--protocol", protocol, "--base_dir", base_dir, "--model_path", model_path]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    match = re.search(r"Final EER:\s*([\d.]+)%", out)
    return float(match.group(1)) if match else None

other_models = {
    "SSL-backend": ("eval_ssl_backend.py", "exp_ssl_backend_full/best.pth"),
    "CNN (spectrogram)": ("eval_cnn_spectrogram.py", "exp_cnn_spectrogram_full/best.pth"),
    "Whisper-backend": ("eval_whisper_backend.py", "exp_whisper_backend_full/best.pth"),
}

for model_label, (script, model_path) in other_models.items():
    for ds_label, (protocol, base_dir) in datasets.items():
        print(f"Running: {model_label} on {ds_label}...")
        eer = run_generic_eval(script, model_path, protocol, base_dir)
        results[(model_label, ds_label)] = eer
        print(f"  -> EER: {eer}%")

print("\nAll runs complete.")

Running: SSL-backend on CSALT (TTS)...
  -> EER: 37.86%
Running: SSL-backend on OpenVoice (own test_audio)...
  -> EER: 46.88%
Running: SSL-backend on OpenVoice (CSALT-converted)...
  -> EER: 47.47%
Running: SSL-backend on MMS-TTS + own recordings...
  -> EER: 67.5%
Running: CNN (spectrogram) on CSALT (TTS)...
  -> EER: 4.16%
Running: CNN (spectrogram) on OpenVoice (own test_audio)...
  -> EER: 21.88%
Running: CNN (spectrogram) on OpenVoice (CSALT-converted)...
  -> EER: 3.39%
Running: CNN (spectrogram) on MMS-TTS + own recordings...
  -> EER: 74.17%
Running: Whisper-backend on CSALT (TTS)...
  -> EER: 16.95%
Running: Whisper-backend on OpenVoice (own test_audio)...
  -> EER: 68.75%
Running: Whisper-backend on OpenVoice (CSALT-converted)...
  -> EER: 47.47%
Running: Whisper-backend on MMS-TTS + own recordings...
  -> EER: 90.42%

All runs complete.


In [ ]:
import pandas as pd

all_models = list(aasist_configs.keys()) + list(other_models.keys())

csalt_only = pd.DataFrame(index=all_models, columns=["CSALT (TTS)"])
for model_label in all_models:
    eer = results.get((model_label, "CSALT (TTS)"))
    csalt_only.loc[model_label, "CSALT (TTS)"] = f"{eer:.2f}%" if eer is not None else "N/A"

print(csalt_only.to_string())

csalt_only.to_csv("/content/drive/MyDrive/deepfake audio detection/csalt_results_table.csv")
print("\nSaved to Drive as csalt_results_table.csv")

                             CSALT (TTS)
AASIST-L Zero-shot                16.09%
AASIST-L Fine-tuned (TTS)         23.98%
AASIST-L Fine-tuned (TTS+VC)      23.61%
SSL-backend                       37.86%
CNN (spectrogram)                  4.16%
Whisper-backend                   16.95%

Saved to Drive as csalt_results_table.csv


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import subprocess
import re
import os

model_registry = {
    "AASIST-L (Zero-shot)": ("aasist", "aasist/models/weights/AASIST-L.pth"),
    "AASIST-L (Fine-tuned TTS)": ("aasist", "exp_urdu_full_v2/best.pth"),
    "AASIST-L (Fine-tuned TTS+VC)": ("aasist", "exp_urdu_vc_included/best.pth"),
    "SSL-backend (wav2vec2)": ("ssl", "exp_ssl_backend_full/best.pth"),
    "CNN (spectrogram)": ("cnn", "exp_cnn_spectrogram_full/best.pth"),
    "Whisper-backend": ("whisper", "exp_whisper_backend_full/best.pth"),
}

model_dropdown = widgets.Dropdown(
    options=list(model_registry.keys()),
    description="Model:",
    style={'description_width': 'initial'}
)

upload_button = widgets.FileUpload(accept='.wav,.mp3,.flac', multiple=False, description="Upload audio")
run_button = widgets.Button(description="Run Detection", button_style='success')
output_area = widgets.Output()

def run_detection(b):
    with output_area:
        clear_output()
        if not upload_button.value:
            print("Please upload an audio file first.")
            return

        uploaded_filename = list(upload_button.value.keys())[0]
        file_content = upload_button.value[uploaded_filename]['content']
        local_path = f"/content/work/{uploaded_filename}"
        with open(local_path, "wb") as f:
            f.write(file_content)
        print(f"Saved: {local_path}")

        model_type, model_path = model_registry[model_dropdown.value]
        print(f"Running: {model_dropdown.value}...")

        cmd = ["python", "detect_single.py", "--audio", local_path,
               "--model_type", model_type, "--model_path", model_path]
        result = subprocess.run(cmd, capture_output=True, text=True, cwd="/content/work")
        print(result.stdout)
        if result.returncode != 0:
            print("ERROR:", result.stderr)

run_button.on_click(run_detection)

display(widgets.VBox([
    widgets.HTML("<h3>Deepfake Audio Detector — Live Demo</h3>"),
    model_dropdown,
    upload_button,
    run_button,
    output_area
]))